# TO DO description and run on google colab

## Imports

In [1]:
from fame.experiments import exp_A_1, exp_A_2, exp_A_2_no_overwrite, exp_B_no_overwrite
from keras.datasets import mnist
from keras.models import load_model
from fame.abstract_domain.utils import check_is_robust 
import numpy as np

/Users/ducoffe_m/Library/Python/3.9/lib/python/site-packages/onnxscript/converter.py:820: FutureWarning: 'onnxscript.values.Op.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/Users/ducoffe_m/Library/Python/3.9/lib/python/site-packages/onnxscript/converter.py:820: FutureWarning: 'onnxscript.values.OnnxFunction.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()


## Experiments setup for MNIST cnn

In [2]:
DATASET = "MNIST"
MODEL = "cnn"
eps = 0.05

channel = 1
data_format = "channels_first"
n_class = 10

In [3]:
## load the model and data

k_model = load_model("./models/xairobas_mnist-cnn.keras")


"""
download and process MNIST data.
"""
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(x_train.shape[0], 28 * 28)
x_test = x_test.reshape(x_test.shape[0], 28 * 28)
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

X = np.concatenate([x_train, x_test])
Y = np.concatenate([y_train, y_test])

In [4]:
k_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ nist-cnn_conv2d_BiasAdd__6      │ (1, 1, 28, 28)         │             0 │
│ (Reshape)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nist-cnn_conv2d_BiasAdd         │ (1, 4, 13, 13)         │            40 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nist-cnn_conv2d_1_BiasAdd       │ (1, 4, 6, 6)           │           148 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nist-cnn_conv2d_1_BiasAdd__12   │ (1, 6, 6, 4)           │             0 │
│ (Permute)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nist-cnn_flatten_Reshape        │ (1, 144)               │             0 │
│ (Reshape)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (1, 20)                │         2,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (1, 20)                │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (1, 10)                │           210 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,298 (12.88 KB)

 Trainable params: 3,298 (12.88 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
indices=[0, 2, 3, 4, 5, 6, 8, 9, 11, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 33, 35, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 52, 53, 54, 55, 57, 58, 59, 60, 61, 62, 64, 65, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 89, 92, 93, 94, 96, 97, 98, 99]

In [6]:
def is_robust(j):
    return check_is_robust(model=k_model, 
                    input_sample=X[j], 
                    eps=eps, 
                    channel=channel, 
                    data_format=data_format, 
                    n_class=n_class)

In [7]:
indices = [i for i in indices if not is_robust(i)]

In [8]:
len(indices)

67

In [9]:
dataframe_repository = "./results"

## Experiment A: MILP versus Greedy

### A.1 in one round: what is the running time and largest free set obtained when using milp or greedy

In [10]:
EXP = "A_1"
filename = "{}_{}_{}".format(DATASET, MODEL, EXP)
dataframe_filename_A1 = filename

In [11]:
for i in range(1):
    dico_a_1 = exp_A_1(
        model=k_model,
        x_test=x_test,
        y_test=y_test,
        indices=indices,
        eps=eps,
        dataframe_repository=dataframe_repository,
        dataframe_filename="{}_v{}".format(dataframe_filename_A1, i),
        channel=channel,
        data_format=data_format,
        n_class=n_class,
        verbose=1,
    )

ongoing index 0
Restricted license - for non-production use only - expires 2026-11-23
milp time 7.523260831832886
greedy time 0.6664340496063232
ongoing index 2
milp time 8.769562005996704
greedy time 0.5832297801971436
ongoing index 3
milp time 0.4886598587036133
greedy time 0.46762871742248535
ongoing index 4
milp time 7.697211980819702
greedy time 0.5073659420013428
ongoing index 5
milp time 7.438499212265015
greedy time 0.5620667934417725
ongoing index 6
milp time 9.944397211074829
greedy time 0.5685560703277588
ongoing index 8
milp time 0.4234342575073242
greedy time 0.42392992973327637
ongoing index 9
milp time 7.963896036148071
greedy time 0.5172379016876221
ongoing index 11
milp time 8.925952911376953
greedy time 0.5666937828063965
ongoing index 14
milp time 5.569894075393677
greedy time 0.5058779716491699
ongoing index 17
milp time 5.9774229526519775
greedy time 0.5030262470245361
ongoing index 18
milp time 7.280452013015747
greedy time 0.5164659023284912
ongoing index 19
milp

### A.2 iteratively, what is the running time and largest free set obtained when using milp or greedy

In [ ]:
EXP = "A_2"
filename = "{}_{}_{}".format(DATASET, MODEL, EXP)
dataframe_filename_A2 = filename

In [ ]:
i = 0
for j in range(len(indices)):
    dico_a_2 = exp_A_2_no_overwrite(
        model=k_model,
        x_test=x_test,
        y_test=y_test,
        indices=indices[j : j + 1],
        eps=eps,
        dataframe_repository=dataframe_repository,
        dataframe_filename="{}_v{}".format(dataframe_filename_A2, i),
        channel=channel,
        data_format=data_format,
        n_class=n_class,
        verbose=1,
        sleep_time=0,
    )

In [ ]:
indices[j : j + 1]

# B Distance 2 minimality

In [ ]:
EXP = "B"
filename = "{}_{}_{}".format(DATASET, MODEL, EXP)
dataframe_filename_B = filename

In [ ]:
for j in indices:
    print("ongoing index", j)
    exp_B_no_overwrite(
        model=k_model,
        x_test=x_test,
        y_test=y_test,
        indices=[j],
        eps=eps,
        dataframe_repository=dataframe_repository,
        dataframe_filename=dataframe_filename_B,
        channel=channel,
        data_format=data_format,
        method="greedy",
        attack="fgsm",
        traversal_order="greedy",
        device="mps",
        n_class=n_class,
    )